## Acquiring and downloading Sentinel-1, Sentinel-2 & Landsat


Same cell as Notebook 1. Run it first.

In [ ]:
import ee
import geemap
import os

PROJECT_ID = "riftwaters"
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

ROI = ee.Geometry.Polygon([
            [36.244378206729166,-0.838827881647561],
            [36.442132112979166,-0.838827881647561],
            [36.442132112979166,-0.6664947946163098],
            [36.244378206729166,-0.6664947946163098],
            [36.244378206729166,-0.838827881647561]     
        ])

START_DATE = "2026-01-01"
END_DATE = "2026-02-28"
REGION_NAME = "naivasha"
# image_id convention used across the whole project: 8-digit int from end_date
IMAGE_ID = int(END_DATE.replace("-", ""))

OUTPUT_DIR = f"./training/{REGION_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("image_id for this run:", IMAGE_ID)

## 1. Sentinel-2

S2 is the most intuitive starting point — optical bands, straightforward cloud masking via the `QA60` bitmask, and a familiar RGB composite.

In [ ]:
s2_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ROI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", 30))
)

count = s2_collection.size().getInfo()
print(f"Found {count} Sentinel-2 scenes")

**Cloud masking**: `QA60` bit 10 = opaque clouds, bit 11 = cirrus clouds. We mask both out before compositing.

In [ ]:
def mask_s2_clouds(image):
    qa = image.select("QA60")
    cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(cloud_mask)

s2_masked = s2_collection.map(mask_s2_clouds)
s2_composite = s2_masked.median().clip(ROI)

# Rename to human-readable band names — used consistently across the project
bands = ["B2", "B3", "B4", "B8", "B11", "B12"]
band_names = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]
s2_composite = s2_composite.select(bands, band_names)

print("S2 composite bands:", s2_composite.bandNames().getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(ROI, 12)
Map.addLayer(s2_composite, {"bands": ["Red", "Green", "Blue"], "min": 0, "max": 3000}, "Sentinel-2")
Map

**Export — local**: `geemap.ee_export_image` downloads directly to disk. It's convenient, but it **silently fails on large exports** — it can return without error while writing a truncated or empty file. Always verify.

In [ ]:
s2_path = f"{OUTPUT_DIR}/{REGION_NAME}_{IMAGE_ID}_s2.tif"

geemap.ee_export_image(
    s2_composite,
    filename=s2_path,
    scale=100,   # coarse scale for training — fast downloads
    region=ROI,
)

# Never trust the call above on its own — check the file actually landed
assert os.path.exists(s2_path) and os.path.getsize(s2_path) > 0, "Export failed or produced an empty file"
print(f"✅ Verified: {s2_path} ({os.path.getsize(s2_path)} bytes)")

## 2. Sentinel-1 (SAR)

SAR is different in three ways that trip people up coming from optical imagery:

1. No clouds to mask — SAR penetrates cloud cover, which is *why* we use it for flood mapping. Instead we filter by **orbit pass** and **polarization**.
2. Raw values are backscatter (dB), not reflectance — the "edge" pixels near swath boundaries are unreliable and get masked out.


In [ ]:
s1_collection = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(ROI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))
    .filter(ee.Filter.eq("instrumentMode", "IW"))
)

count = s1_collection.size().getInfo()
print(f"Found {count} Sentinel-1 scenes")

Edge masking removes the unreliable low-signal pixels near swath boundaries before compositing.

In [ ]:
def preprocess_sar(image):
    edge = image.lt(-30.0)
    masked = image.mask().And(edge.Not())
    return image.updateMask(masked).clip(ROI)

s1_processed = s1_collection.map(preprocess_sar)
s1_composite = s1_processed.median()

print("S1 composite bands:", s1_composite.bandNames().getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(ROI, 12)
Map.addLayer(s1_composite, {"bands": ["VV"], "min": -25, "max": 0}, "Sentinel-1 (VV)")
Map

In [ ]:
s1_path = f"{OUTPUT_DIR}/{REGION_NAME}_{IMAGE_ID}_VV_ASCENDING.tif"

geemap.ee_export_image(
    s1_composite,
    filename=s1_path,
    scale=10,
    region=ROI,
)

assert os.path.exists(s1_path) and os.path.getsize(s1_path) > 0, "Export failed or produced an empty file"
print(f"✅ Verified: {s1_path} ({os.path.getsize(s1_path)} bytes)")

## 3. Landsat 8 / 9

Landsat Collection 2 Level-2 products are delivered as **scaled integers**, not physical units — you must apply the documented scale factors before the values mean anything (reflectance, temperature).

In [ ]:
landsat_collection = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(ROI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUD_COVER", 30))
)

count = landsat_collection.size().getInfo()
print(f"Found {count} Landsat 8 scenes")

In [ ]:
def scale_landsat(image):
    optical_bands = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]
    optical_names = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]
    thermal_bands = ["ST_B10"]
    thermal_names = ["Thermal"]

    for i, band in enumerate(optical_bands):
        image = image.updateMask(image.select(band).gt(0))
        scaled = image.select(band).multiply(0.0000275).add(-0.2)
        image = image.addBands(scaled.rename(optical_names[i]))

    for i, band in enumerate(thermal_bands):
        scaled = image.select(band).multiply(0.00341802).add(149.0)
        image = image.addBands(scaled.rename(thermal_names[i]))

    return image.select(optical_names + thermal_names)

landsat_scaled = landsat_collection.map(scale_landsat)
landsat_composite = landsat_scaled.median().clip(ROI)

print("Landsat composite bands:", landsat_composite.bandNames().getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(ROI, 12)
Map.addLayer(landsat_composite, {"bands": ["Red", "Green", "Blue"], "min": 0, "max": 0.3}, "Landsat 8")
Map

In [ ]:
landsat_path = f"{OUTPUT_DIR}/{REGION_NAME}_{IMAGE_ID}_landsat8.tif"

geemap.ee_export_image(
    landsat_composite,
    filename=landsat_path,
    scale=30,
    region=ROI,
)

assert os.path.exists(landsat_path) and os.path.getsize(landsat_path) > 0, "Export failed or produced an empty file"
print(f"✅ Verified: {landsat_path} ({os.path.getsize(landsat_path)} bytes)")

## 4. The two export paths — when to use which

| | `geemap.ee_export_image` | `Export.image.toDrive` |
|---|---|---|
| Where it lands | Local disk, immediately | Your Google Drive, async |
| Size limit | Fails silently above ~a few hundred MB | Handles large exports (up to `maxPixels`) |
| Speed | Synchronous — blocks until done or silently truncated | Async — submit task, poll for completion |
| When to use | Small ROIs, quick iteration, training | Production-scale exports |

For anything beyond a small training ROI, the safe pattern is: try the local export, verify the file, and fall back to a Drive export + polling loop if it failed. We won't build the polling/fallback logic today — just know it exists for when you scale beyond a training ROI.

## Exercise 2.1

Pick a lake (ROI below) and a **different one-month date range** within 2023–2024.

1. Acquire and export a Sentinel-2 composite.
2. Acquire and export a Sentinel-1 (VV, ASCENDING) composite
3. Verify both files exist and are non-empty before moving on.

```python
# Nakuru ROI, for reference
NAKURU_ROI = ee.Geometry.Polygon([
    [36.045150257654164, -0.41882934935973415],
    [36.13441417366979, -0.41882934935973415],
    [36.13441417366979, -0.29935539913088666],
    [36.045150257654164, -0.29935539913088666],
    [36.045150257654164, -0.41882934935973415],
])

ELEMENTAITA_ROI = ee.Geometry.Polygon([
            [36.20503005704847,-0.4844042871968538],
            [36.27541122159925,-0.4844042871968538],
            [36.27541122159925,-0.3992627836576272],
            [36.20503005704847,-0.3992627836576272],
            [36.20503005704847,-0.4844042871968538]
        ])
BARINGO_ROI = ee.Geometry.Polygon([
            [35.98987673288601,0.4361077040885743],
            [36.180764184057885,0.4361077040885743],
            [36.180764184057885,0.7574399496732773],
            [35.98987673288601,0.7574399496732773],
            [35.98987673288601,0.4361077040885743]
        ])

```


In [ ]:
# Your solution here
